In [1]:
import pandas as pd
import re
from collections import Counter


In [2]:

resume_df = pd.read_csv("../data/processed/resumes_clean.csv")
skills_df = pd.read_csv("../data/processed/skills_clean.csv")

print("Resumes:", len(resume_df))
print("Skills:", len(skills_df))

Resumes: 2484
Skills: 36167


In [3]:
resume_text = resume_df["clean_resume"].fillna("")
skill_vocabulary = skills_df["clean_skill"].tolist()

print("Total resumes:", len(resume_text))
print("Total skills:", len(skill_vocabulary))

Total resumes: 2484
Total skills: 36167


In [4]:
# Combine all resumes into one text
all_resumes_text = " ".join(resume_text)

print("Combined resume text created.")
print("Total characters:", len(all_resumes_text))

Combined resume text created.
Total characters: 14712248


In [5]:
test_vocabulary = skill_vocabulary[:500]

print("Testing with:", len(test_vocabulary), "skills")

Testing with: 500 skills


In [6]:
test_counts = {}

for skill in test_vocabulary:
    pattern = r"(?<!\w)" + re.escape(skill) + r"(?!\w)"
    count = len(re.findall(pattern, all_resumes_text, flags=re.IGNORECASE))
    test_counts[skill] = count

print("Test completed!")

Test completed!


In [7]:
test_results = pd.DataFrame(
    test_counts.items(),
    columns=["clean_skill", "resume_count"]
)

test_results = test_results.sort_values(
    "resume_count",
    ascending=False
)

test_results.head(30)

,clean_skill,resume_count
97,maintenance,1824
299,cash,1685
438,press releases,209
263,computer science,183
269,french,127
87,information system,99
41,ethics,81
76,prototype,74
35,startup,68
388,prospect,63


In [8]:
important_skills = [
    "python",
    "java",
    "javascript",
    "c++",
    "c#",
    "sql",
    "html",
    "css",
    "react",
    "react.js",
    "node.js",
    "mongodb",
    "mysql",
    "postgresql",
    "numpy",
    "pandas",
    "scikit-learn",
    "tensorflow",
    "pytorch",
    "machine learning",
    "deep learning",
    "artificial intelligence",
    "data analysis",
    "data analytics",
    "power bi",
    "tableau",
    "git",
    "github"
]

for skill in important_skills:
    matches = skills_df[
        skills_df["clean_skill"].str.lower() == skill.lower()
    ]

    if len(matches) > 0:
        print(f"✓ {skill}")
    else:
        print(f"✗ {skill}")

✓ python
✓ java
✓ javascript
✓ c++
✓ c#
✓ sql
✓ html
✓ css
✗ react
✓ react.js
✓ node.js
✓ mongodb
✓ mysql
✓ postgresql
✓ numpy
✓ pandas
✓ scikit-learn
✓ tensorflow
✓ pytorch
✓ machine learning
✓ deep learning
✓ artificial intelligence
✓ data analysis
✓ data analytics
✓ power bi
✓ tableau
✓ git
✓ github


In [9]:
test_skills = [
    "python",
    "java",
    "javascript",
    "c++",
    "c#",
    "sql",
    "html",
    "css",
    "react.js",
    "node.js",
    "mongodb",
    "mysql",
    "postgresql",
    "numpy",
    "pandas",
    "scikit-learn",
    "tensorflow",
    "pytorch",
    "machine learning",
    "deep learning",
    "artificial intelligence",
    "data analysis",
    "data analytics",
    "power bi",
    "tableau",
    "git",
    "github"
]

print("Test vocabulary size:", len(test_skills))

Test vocabulary size: 27


In [10]:
# Testing on sample resume

In [11]:
sample_resume = """
I am a Data Analyst with experience in Python, Pandas, NumPy and SQL.
I have worked on machine learning projects using Scikit-learn and
TensorFlow. I also have experience with Power BI, Tableau, Git and GitHub.
"""

sample_resume = sample_resume.lower()

print(sample_resume)


i am a data analyst with experience in python, pandas, numpy and sql.
i have worked on machine learning projects using scikit-learn and
tensorflow. i also have experience with power bi, tableau, git and github.



In [12]:
def extract_skills(text, skills):
    text = text.lower()
    found_skills = []

    for skill in skills:
        pattern = r"(?<!\w)" + re.escape(skill.lower()) + r"(?!\w)"

        if re.search(pattern, text):
            found_skills.append(skill)

    return found_skills

In [13]:
detected_skills = extract_skills(
    sample_resume,
    test_skills
)

print("Detected skills:")
print(detected_skills)

Detected skills:
['python', 'sql', 'numpy', 'pandas', 'scikit-learn', 'tensorflow', 'machine learning', 'power bi', 'tableau', 'git', 'github']


In [14]:
# Testing on Real Resume

In [15]:
real_resume = resume_df["clean_resume"].iloc[0]

print(real_resume[:2000])

hr administrator/marketing associate hr administrator summary dedicated customer service manager with 15+ years of experience in hospitality and customer service management. respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service. highlights focused on customer satisfaction team management marketing savvy conflict resolution techniques training and development skilled multi-tasker client relations specialist accomplishments missouri dot supervisor training certification certified by ihg in customer loyalty and marketing by segment hilton worldwide general manager training certification accomplished trainer for cross server hospitality systems such as hilton onq , micros opera pms , fidelio opera reservation system (ors) , holidex completed courses and seminars in customer service, sales strategies, inventory control, loss prevention, safety, time management, leadership and performance assessment. experience hr adm

In [16]:
real_detected_skills = extract_skills(
    real_resume,
    test_skills
)

print("Detected skills:")
print(real_detected_skills)

Detected skills:
['data analysis']


In [17]:
# Building extractor for Resumes

In [18]:
# Create a set of all normalized skills
skill_set = set(skill_vocabulary)

print("Total skills in lookup:", len(skill_set))

Total skills in lookup: 36167


In [19]:
# Create a faster extractor

In [20]:
def extract_skills_fast(text, skill_set):
    text = text.lower()
    
    # Keep technical characters such as +, #, ., -
    text = re.sub(r'[^a-z0-9+#.\- ]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    words = text.split()
    
    found_skills = set()
    
    # Check phrases from longest to shortest
    max_words = 5
    
    for n in range(max_words, 0, -1):
        for i in range(len(words) - n + 1):
            phrase = " ".join(words[i:i+n])
            
            if phrase in skill_set:
                found_skills.add(phrase)
    
    return sorted(found_skills)

In [21]:
# Testing on Real resumes

In [22]:
real_detected_skills = extract_skills_fast(
    real_resume,
    skill_set
)

print("Detected skills:")
print(real_detected_skills)

Detected skills:
['2004', '2007', '2010', '2012', '2013', 'accounting', 'administration', 'advanced', 'advertising', 'analysis', 'analytical', 'analytical skills', 'billing', 'budgeting', 'certification', 'codes', 'collateral', 'compensation', 'computer skills', 'conflict', 'conflict resolution', 'customer', 'customer loyalty', 'customer satisfaction', 'customer service', 'daily operations', 'data', 'data analysis', 'dental insurance', 'design', 'documentation', 'employee relations', 'executive support', 'facebook', 'features', 'financial management', 'front office', 'front-end', 'general', 'government', 'government relations', 'health', 'healthcare', 'holidex', 'hospital', 'hospitality', 'hotel', 'human resources', 'icd-9', 'insurance', 'interpret', 'inventory', 'inventory control', 'labor relations', 'layout', 'leadership', 'leave of absence', 'loss prevention', 'management', 'management accounting', 'marketing', 'medical billing', 'medical terminology', 'micros', 'morale', 'office',

In [23]:
skills_extraction = skills_df.copy()

skills_extraction = skills_extraction[
    ~skills_extraction["clean_skill"].str.fullmatch(r"\d+")
].reset_index(drop=True)

print("Original skills:", len(skills_df))
print("After removing numeric entries:", len(skills_extraction))

Original skills: 36167
After removing numeric entries: 36149


In [24]:
protected_skills = {
    "python",
    "java",
    "javascript",
    "c",
    "c++",
    "c#",
    "f#",
    "g++",
    "sql",
    "html",
    "css",
    "php",
    "ruby",
    "swift",
    "kotlin",
    "go",
    "r",
    "matlab",
    "scala",
    "perl",
    "bash",
    "powershell",
    "git",
    "github",
    "gitlab",
    "npm",
    "node.js",
    "react.js",
    "vue",
    "angular",
    "mongodb",
    "mysql",
    "postgresql",
    "oracle",
    "redis",
    "numpy",
    "pandas",
    "tensorflow",
    "pytorch",
    "keras",
    "tableau",
    "power bi",
    "docker",
    "kubernetes",
    "aws",
    "azure",
    "gcp",
    "linux",
    "android",
    "ios",
    "sap",
    "excel",
    "jira"
}

print("Protected skills:", len(protected_skills))

Protected skills: 53


In [25]:
generic_words = {
    "advanced",
    "analysis",
    "analytical",
    "codes",
    "collateral",
    "conflict",
    "customer",
    "data",
    "design",
    "features",
    "general",
    "government",
    "health",
    "hospital",
    "hotel",
    "insurance",
    "interpret",
    "layout",
    "management",
    "office",
    "operations",
    "paperwork",
    "participation",
    "plan",
    "policies",
    "prevention",
    "print",
    "public",
    "ratings",
    "records",
    "reporting",
    "reviews",
    "safety",
    "sales",
    "service",
    "skills",
    "statistics",
    "system",
    "teams",
    "terminology",
    "training",
    "transactions",
    "website",
    "customer",
    "organization",
    "professional",
    "responsibilities",
    "experience",
    "work",
    "working"
}

print("Generic words:", len(generic_words))

Generic words: 49


In [26]:
def is_valid_extraction_skill(skill):
    skill = skill.lower().strip()

    # Keep multi-word skills
    if len(skill.split()) > 1:
        return True

    # Keep protected technical skills
    if skill in protected_skills:
        return True

    # Remove generic standalone words
    if skill in generic_words:
        return False

    return True

In [27]:
skills_extraction = skills_extraction[
    skills_extraction["clean_skill"].apply(is_valid_extraction_skill)
].reset_index(drop=True)

print("Final extraction vocabulary:", len(skills_extraction))

Final extraction vocabulary: 36106


In [28]:
skill_vocabulary = skills_extraction["clean_skill"].tolist()

skill_set = set(skill_vocabulary)

print("Final extraction vocabulary:", len(skill_set))

Final extraction vocabulary: 36106


In [29]:
def extract_skills(text, skill_set, max_words=5):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9+#.\- ]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    words = text.split()
    words = [w.strip('.') for w in words]
    words = [w for w in words if w]

    found_skills = set()

    for n in range(max_words, 0, -1):
        for i in range(len(words) - n + 1):
            phrase = " ".join(words[i:i+n])
            if phrase in skill_set:
                found_skills.add(phrase)

    return sorted(found_skills)

In [30]:
sample_resume = resume_df.iloc[0]["clean_resume"]

extracted_skills = extract_skills(
    sample_resume,
    skill_set
)

print("Number of skills found:", len(extracted_skills))
print("\nSkills found:")
print(extracted_skills)

Number of skills found: 61

Skills found:
['accounting', 'administration', 'advertising', 'analytical skills', 'assessment', 'billing', 'budgeting', 'certification', 'commerce', 'compensation', 'computer skills', 'conflict resolution', 'customer loyalty', 'customer satisfaction', 'customer service', 'customer service management', 'daily operations', 'data analysis', 'dental insurance', 'documentation', 'employee relations', 'executive support', 'facebook', 'financial management', 'front office', 'front-end', 'government relations', 'healthcare', 'holidex', 'hospitality', 'human resources', 'icd-9', 'inventory', 'inventory control', 'labor relations', 'leadership', 'leave of absence', 'loss prevention', 'management accounting', 'marketing', 'medical billing', 'medical terminology', 'micros', 'morale', 'opera', 'opera reservation system', 'payroll', 'pension', 'performance reviews', 'presentations', 'productivity', 'programming', 'public relations', 'purchasing', 'service delivery', 'ski

In [31]:
def remove_overlapping_skills(skills):
    skills = sorted(skills, key=lambda x: len(x.split()), reverse=True)
    final_skills = []

    for skill in skills:
        skill_words = skill.split()
        is_subskill = False

        for existing_skill in final_skills:
            existing_words = existing_skill.split()
            n = len(skill_words)
            # check if skill_words appears as a contiguous run inside existing_words
            for i in range(len(existing_words) - n + 1):
                if existing_words[i:i+n] == skill_words:
                    is_subskill = True
                    break
            if is_subskill:
                break

        if not is_subskill:
            final_skills.append(skill)

    return sorted(final_skills)

In [32]:
final_skills = remove_overlapping_skills(extracted_skills)

print("Number of final skills:", len(final_skills))
print("\nFinal skills:")
print(final_skills)

Number of final skills: 56

Final skills:
['administration', 'advertising', 'analytical skills', 'assessment', 'budgeting', 'certification', 'commerce', 'compensation', 'computer skills', 'conflict resolution', 'customer loyalty', 'customer satisfaction', 'customer service management', 'daily operations', 'data analysis', 'dental insurance', 'documentation', 'employee relations', 'executive support', 'facebook', 'financial management', 'front office', 'front-end', 'government relations', 'healthcare', 'holidex', 'hospitality', 'human resources', 'icd-9', 'inventory control', 'labor relations', 'leadership', 'leave of absence', 'loss prevention', 'management accounting', 'marketing', 'medical billing', 'medical terminology', 'micros', 'morale', 'opera reservation system', 'payroll', 'pension', 'performance reviews', 'presentations', 'productivity', 'programming', 'public relations', 'purchasing', 'service delivery', 'skilled multi-tasker', 'supervisor', 'swift', 'team management', 'time

In [33]:
def extract_final_skills(text, skill_set, max_words=5):
    
    # Step 1: Normalize aliases before extraction
    normalized_text = normalize_text_aliases(
        text,
        skill_aliases
    )

    # Step 2: Extract skills
    extracted = extract_skills(
        normalized_text,
        skill_set,
        max_words=max_words
    )

    # Step 3: Normalize extracted skills
    normalized = [
        normalize_skill(skill)
        for skill in extracted
    ]

    # Step 4: Remove duplicates
    normalized = list(set(normalized))

    # Step 5: Remove overlapping skills
    final = remove_overlapping_skills(normalized)

    return final

In [58]:
final_skills = extract_final_skills(
    sample_resume,
    skill_set
)

print("Number of final skills:", len(final_skills))
print(final_skills)

Number of final skills: 56
['administration', 'advertising', 'analytical skills', 'assessment', 'budgeting', 'certification', 'commerce', 'compensation', 'computer skills', 'conflict resolution', 'customer loyalty', 'customer satisfaction', 'customer service management', 'daily operations', 'data analysis', 'dental insurance', 'documentation', 'employee relations', 'executive support', 'facebook', 'financial management', 'front office', 'front-end', 'government relations', 'healthcare', 'holidex', 'hospitality', 'human resources', 'icd-9', 'inventory control', 'labor relations', 'leadership', 'leave of absence', 'loss prevention', 'management accounting', 'marketing', 'medical billing', 'medical terminology', 'micros', 'morale', 'opera reservation system', 'payroll', 'pension', 'performance reviews', 'presentations', 'productivity', 'programming', 'public relations', 'purchasing', 'service delivery', 'skilled multi-tasker', 'supervisor', 'swift', 'team management', 'time management', '

In [57]:
technical_test = """
Software developer with experience in Python, Java, C++, C#, 
JavaScript, React.js, Node.js, SQL, MongoDB, PostgreSQL,
NumPy, Pandas, Scikit-learn, TensorFlow, PyTorch,
Machine Learning, Deep Learning, Data Analysis,
Power BI, Tableau, Git, GitHub, Docker, AWS and SVG.
"""

technical_skills = extract_final_skills(
    technical_test,
    skill_set
)

print("Number of skills:", len(technical_skills))
print("\nDetected skills:")

for skill in technical_skills:
    print("-", skill)

Number of skills: 24

Detected skills:
- aws
- c#
- c++
- data analysis
- deep learning
- docker
- git
- github
- java
- javascript
- machine learning
- mongodb
- node.js
- numpy
- pandas
- postgresql
- power bi
- python
- pytorch
- scikit-learn
- sql
- svg
- tableau
- tensorflow


In [43]:
print("svg" in skill_set)
skills_df[skills_df["clean_skill"].str.lower() == "svg"]

True


,Skill,clean_skill,skill_length,word_count
4099,svg\n,svg,3,1


In [44]:
generic_words = {
    "advanced", "analysis", "analytical", "codes", "collateral",
    "conflict", "customer", "data", "design", "features",
    "general", "government", "health", "hospital", "hotel",
    "insurance", "interpret", "layout", "management", "office",
    "operations", "paperwork", "participation", "plan", "policies",
    "prevention", "print", "public", "ratings", "records",
    "reporting", "reviews", "safety", "sales", "service",
    "skills", "statistics", "system", "teams", "terminology",
    "training", "transactions", "website", "organization",
    "professional", "responsibilities", "experience", "work",
    "working", "software"
}

In [45]:
def is_valid_extraction_skill(skill):
    skill = skill.lower().strip()

    if len(skill.split()) > 1:
        return True

    if skill in protected_skills:
        return True

    if skill in generic_words:
        return False

    return True


# Re-apply numeric-entry filter (years, IDs) before rebuilding skills_extraction,
# otherwise it silently comes back since we're starting again from skills_df here.
skills_extraction = skills_df[
    ~skills_df["clean_skill"].str.fullmatch(r"\d+")
]

skills_extraction = skills_extraction[
    skills_extraction["clean_skill"].apply(is_valid_extraction_skill)
].copy()

skills_extraction = skills_extraction.reset_index(drop=True)

skill_vocabulary = skills_extraction["clean_skill"].tolist()
skill_set = set(skill_vocabulary)

print("Final extraction vocabulary:", len(skill_vocabulary))

Final extraction vocabulary: 36105


In [56]:
technical_skills = extract_final_skills(
    technical_test,
    skill_set
)

print("Number of skills:", len(technical_skills))
print("\nDetected skills:")

for skill in technical_skills:
    print("-", skill)

Number of skills: 24

Detected skills:
- aws
- c#
- c++
- data analysis
- deep learning
- docker
- git
- github
- java
- javascript
- machine learning
- mongodb
- node.js
- numpy
- pandas
- postgresql
- power bi
- python
- pytorch
- scikit-learn
- sql
- svg
- tableau
- tensorflow


In [47]:
skill_aliases = {

    # Frontend
    "react": "react.js",
    "reactjs": "react.js",
    "react js": "react.js",

    "vuejs": "vue",
    "vue js": "vue",

    "angularjs": "angular",
    "angular js": "angular",

    # Backend
    "nodejs": "node.js",
    "node js": "node.js",

    # Machine Learning / AI
    "scikit learn": "scikit-learn",
    "scikit learn library": "scikit-learn",

    "tensorflow library": "tensorflow",

    "pytorch library": "pytorch",

    "ml": "machine learning",
    "ai": "artificial intelligence",

    # Data
    "data analytics": "data analysis",

    # Databases
    "mongo": "mongodb",
    "mongo db": "mongodb",

    "postgre sql": "postgresql",

    # Programming languages
    "c sharp": "c#",
    "cpp": "c++",

    # Cloud
    "amazon web services": "aws",
    "microsoft azure": "azure",
    "google cloud platform": "gcp",

    # Version control
    "git hub": "github",

    # Power BI
    "powerbi": "power bi"
}

In [48]:
def normalize_skill(skill):
    skill = skill.lower().strip()

    if skill in skill_aliases:
        return skill_aliases[skill]

    return skill

In [49]:
test_skills = [
    "React",
    "React.js",
    "ReactJS",
    "React JS",
    "Node.js",
    "NodeJS",
    "Node JS",
    "Scikit Learn",
    "Mongo DB",
    "C Sharp",
    "CPP",
    "PowerBI",
    "Git Hub"
]

for skill in test_skills:
    print(f"{skill:15} → {normalize_skill(skill)}")

React           → react.js
React.js        → react.js
ReactJS         → react.js
React JS        → react.js
Node.js         → node.js
NodeJS          → node.js
Node JS         → node.js
Scikit Learn    → scikit-learn
Mongo DB        → mongodb
C Sharp         → c#
CPP             → c++
PowerBI         → power bi
Git Hub         → github


In [55]:
messy_resume = """
Software Engineer with experience in Python, Java, CPP and C Sharp.
Worked extensively with ReactJS, React JS and Node JS.
Experienced with Mongo DB, Postgre SQL and Scikit Learn.
Used PowerBI and Git Hub for data analysis and version control.
Worked with TensorFlow, PyTorch, Docker, AWS and SVG.
"""

messy_skills = extract_final_skills(
    messy_resume,
    skill_set
)

print("Number of skills:", len(messy_skills))
print("\nDetected and normalized skills:")

for skill in messy_skills:
    print("-", skill)

Number of skills: 18

Detected and normalized skills:
- aws
- c#
- c++
- data analysis
- docker
- github
- java
- mongodb
- node.js
- postgresql
- power bi
- python
- pytorch
- react.js
- scikit-learn
- svg
- tensorflow
- version control


In [53]:
def normalize_text_aliases(text, skill_aliases):
    text = str(text).lower()

    # Single pass: build one alternation regex (longest alias first so it wins
    # at each position) and substitute once against the ORIGINAL text.
    # Looping re.sub() over progressively-mutated text is what caused the bug:
    # a short alias like "react" would re-match inside an already-converted
    # "react.js" (since "." satisfies the word-boundary lookaround), producing
    # "react.js.js" - which then fails to match skill_set and gets silently
    # dropped from extraction entirely. A single pass over the original text
    # can't re-match text it already replaced, so this can't happen.
    aliases_sorted = sorted(skill_aliases.keys(), key=len, reverse=True)
    pattern = re.compile(
        r'(?<![a-z0-9])(' + '|'.join(re.escape(a) for a in aliases_sorted) + r')(?![a-z0-9])'
    )

    return pattern.sub(lambda m: skill_aliases[m.group(0)], text)

In [54]:
normalized_test = normalize_text_aliases(
    messy_resume,
    skill_aliases
)

print(normalized_test)


software engineer with experience in python, java, c++ and c#.
worked extensively with react.js, react.js and node.js.
experienced with mongodb, postgresql and scikit-learn.
used power bi and github for data analysis and version control.
worked with tensorflow, pytorch, docker, aws and svg.



In [59]:
technical_skills_reference = {
    "python",
    "java",
    "javascript",
    "c",
    "c++",
    "c#",
    "f#",
    "php",
    "ruby",
    "swift",
    "kotlin",
    "go",
    "r",
    "matlab",
    "scala",
    "perl",
    "bash",
    "powershell",

    "html",
    "css",
    "react.js",
    "vue",
    "angular",
    "node.js",

    "sql",
    "mysql",
    "postgresql",
    "mongodb",
    "oracle",
    "redis",

    "numpy",
    "pandas",
    "scikit-learn",
    "tensorflow",
    "pytorch",
    "keras",

    "machine learning",
    "deep learning",
    "artificial intelligence",
    "data analysis",
    "data analytics",
    "natural language processing",
    "computer vision",

    "git",
    "github",
    "gitlab",

    "docker",
    "kubernetes",

    "aws",
    "azure",
    "gcp",

    "linux",
    "android",
    "ios",

    "tableau",
    "power bi",
    "excel",
}

In [60]:
technical_detected = [
    skill for skill in final_skills
    if skill in technical_skills_reference
]

other_detected = [
    skill for skill in final_skills
    if skill not in technical_skills_reference
]

print("Technical skills:")
for skill in technical_detected:
    print("-", skill)

print("\nOther detected skills:")
for skill in other_detected:
    print("-", skill)

Technical skills:
- data analysis
- swift

Other detected skills:
- administration
- advertising
- analytical skills
- assessment
- budgeting
- certification
- commerce
- compensation
- computer skills
- conflict resolution
- customer loyalty
- customer satisfaction
- customer service management
- daily operations
- dental insurance
- documentation
- employee relations
- executive support
- facebook
- financial management
- front office
- front-end
- government relations
- healthcare
- holidex
- hospitality
- human resources
- icd-9
- inventory control
- labor relations
- leadership
- leave of absence
- loss prevention
- management accounting
- marketing
- medical billing
- medical terminology
- micros
- morale
- opera reservation system
- payroll
- pension
- performance reviews
- presentations
- productivity
- programming
- public relations
- purchasing
- service delivery
- skilled multi-tasker
- supervisor
- team management
- time management
- trade shows


In [69]:
technical_skills_reference = {
    "python", "java", "javascript", "c", "c++", "c#",
    "f#", "php", "ruby", "swift", "kotlin", "go", "r",
    "matlab", "scala", "perl", "bash", "powershell",

    "html", "css", "react.js", "vue", "angular", "node.js","front-end",

    "sql", "mysql", "postgresql", "mongodb", "oracle", "redis",

    "numpy", "pandas", "scikit-learn", "tensorflow",
    "pytorch", "keras",

    "machine learning", "deep learning",
    "artificial intelligence", "data analysis",
    "data analytics", "natural language processing",
    "computer vision",

    "git", "github", "gitlab",
    "docker", "kubernetes",

    "aws", "azure", "gcp",
    "linux", "android", "ios",

    "tableau", "power bi", "excel"
}


software_tools_reference = {
    "facebook",
    "holidex",
    "micros",
    "opera reservation system",
    "opera",
    "icd-9"
}


professional_domain_reference = {
    "administration",
    "advertising",
    "analytical skills",
    "assessment",
    "budgeting",
    "certification",
    "commerce",
    "compensation",
    "computer skills",
    "conflict resolution",
    "customer loyalty",
    "customer satisfaction",
    "customer service",
    "customer service management",
    "daily operations",
    "dental insurance",
    "documentation",
    "employee relations",
    "executive support",
    "financial management",
    "front office",
    "government relations",
    "healthcare",
    "hospitality",
    "human resources",
    "inventory control",
    "labor relations",
    "leadership",
    "leave of absence",
    "loss prevention",
    "management accounting",
    "marketing",
    "medical billing",
    "medical terminology",
    "morale",
    "payroll",
    "pension",
    "performance reviews",
    "presentations",
    "productivity",
    "programming",
    "public relations",
    "purchasing",
    "service delivery",
    "skilled multi-tasker",
    "supervisor",
    "team management",
    "time management",
    "trade shows"
}

In [70]:
def classify_skill(skill):

    if skill in technical_skills_reference:
        return "technical"

    if skill in software_tools_reference:
        return "software/tool"

    if skill in professional_domain_reference:
        return "professional/domain"

    return "other"

In [71]:
for skill in final_skills:

    print(
        f"{skill:30} → {classify_skill(skill)}"
    )

administration                 → professional/domain
advertising                    → professional/domain
analytical skills              → professional/domain
assessment                     → professional/domain
budgeting                      → professional/domain
certification                  → professional/domain
commerce                       → professional/domain
compensation                   → professional/domain
computer skills                → professional/domain
conflict resolution            → professional/domain
customer loyalty               → professional/domain
customer satisfaction          → professional/domain
customer service management    → professional/domain
daily operations               → professional/domain
data analysis                  → technical
dental insurance               → professional/domain
documentation                  → professional/domain
employee relations             → professional/domain
executive support              → professional/domain
fac

In [72]:
def extract_and_classify_skills(text, skill_set, max_words=5):

    # 1. Normalize aliases in the resume text
    normalized_text = normalize_text_aliases(
        text,
        skill_aliases
    )

    # 2. Extract skills from normalized text
    extracted = extract_skills(
        normalized_text,
        skill_set,
        max_words=max_words
    )

    # 3. Normalize extracted skill names
    normalized = [
        normalize_skill(skill)
        for skill in extracted
    ]

    # 4. Remove duplicates
    normalized = list(set(normalized))

    # 5. Remove overlapping skills
    final_skills = remove_overlapping_skills(
        normalized
    )

    # 6. Classify skills
    classified = {
        "technical": [],
        "software/tool": [],
        "professional/domain": [],
        "other": []
    }

    for skill in final_skills:

        category = classify_skill(skill)

        classified[category].append(skill)

    return final_skills, classified

In [73]:
real_resume = resume_df.iloc[0]["clean_resume"]

final_skills, classified_skills = extract_and_classify_skills(
    real_resume,
    skill_set
)

In [74]:
print("Total skills:", len(final_skills))

print("\nTechnical Skills:")
for skill in classified_skills["technical"]:
    print("-", skill)

print("\nSoftware / Tools:")
for skill in classified_skills["software/tool"]:
    print("-", skill)

print("\nProfessional / Domain:")
for skill in classified_skills["professional/domain"]:
    print("-", skill)

print("\nOther:")
for skill in classified_skills["other"]:
    print("-", skill)

Total skills: 56

Technical Skills:
- data analysis
- front-end
- swift

Software / Tools:
- facebook
- holidex
- icd-9
- micros
- opera reservation system

Professional / Domain:
- administration
- advertising
- analytical skills
- assessment
- budgeting
- certification
- commerce
- compensation
- computer skills
- conflict resolution
- customer loyalty
- customer satisfaction
- customer service management
- daily operations
- dental insurance
- documentation
- employee relations
- executive support
- financial management
- front office
- government relations
- healthcare
- hospitality
- human resources
- inventory control
- labor relations
- leadership
- leave of absence
- loss prevention
- management accounting
- marketing
- medical billing
- medical terminology
- morale
- payroll
- pension
- performance reviews
- presentations
- productivity
- programming
- public relations
- purchasing
- service delivery
- skilled multi-tasker
- supervisor
- team management
- time management
- tra